# Building `SGDRegressor` from Scratch

**Part 1 — `list[dict]` in, `list[dict]` out**
**Part 2 — `DataFrame` in, `DataFrame` out**

---

Scikit-learn's `SGDRegressor` is a linear model trained by **stochastic gradient descent**:
instead of solving for the weights in closed form (like `LinearRegression` does), it looks at
**one sample at a time** and nudges the weights a little in the direction that reduces the error.

We will build it in small steps. Every step is runnable on its own, and each one ends with a
check that it actually does what we claimed.

**Roadmap**

| Step | What we build |
|---|---|
| 1 | The data (as a `list[dict]`) |
| 2 | Turning dicts into a matrix — and the key-order trap |
| 3 | The model: `w · x + b` |
| 4 | Loss and gradient for a **single** sample |
| 5 | One epoch of SGD |
| 6 | Learning-rate schedules |
| 7 | The full `SGDRegressorDict` class |
| 8 | Why unscaled features blow it up |
| 9 | Sanity-check against real `SGDRegressor` |
| 10 | `list[dict]` ↔ `DataFrame` bridge |
| 11 | `DataFrame` input — and the column-order trap |
| 12 | `DataFrame` output |
| 13 | One core, two adapters |

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 3)
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.4.4 | pandas 3.0.2


---
## Step 1 — The data

We generate a small housing dataset with a **known** linear rule, so that later we can check
whether our model actually recovered it:

$$\text{price}_k = 0.15 \cdot \text{sqft} + 10 \cdot \text{bedrooms} - 0.8 \cdot \text{age} + 50 + \varepsilon$$

`price_k` is the price in thousands. The features are stored the way an API would hand them
to you: a **list of dictionaries**, one dict per house.

In [2]:
rng = np.random.default_rng(42)
n = 300

sqft     = rng.uniform(800, 3500, n).round(0)
bedrooms = rng.integers(1, 6, n)
age      = rng.uniform(0, 60, n).round(1)

TRUE_COEF = {"sqft": 0.15, "bedrooms": 10.0, "age": -0.8}
TRUE_INTERCEPT = 50.0

price_k = (TRUE_COEF["sqft"] * sqft
           + TRUE_COEF["bedrooms"] * bedrooms
           + TRUE_COEF["age"] * age
           + TRUE_INTERCEPT
           + rng.normal(0, 15, n))

# ---- the input format we commit to: list of dicts ----
records = [
    {"sqft": float(s), "bedrooms": int(b), "age": float(a)}
    for s, b, a in zip(sqft, bedrooms, age)
]
targets = price_k.tolist()          # a plain list of floats

# train / test split (manual, so nothing is hidden)
idx = rng.permutation(n)
cut = int(0.8 * n)
train_i, test_i = idx[:cut], idx[cut:]

X_train = [records[i] for i in train_i]
y_train = [targets[i] for i in train_i]
X_test  = [records[i] for i in test_i]
y_test  = [targets[i] for i in test_i]

print(f"train: {len(X_train)} rows | test: {len(X_test)} rows")
print("first record:", X_train[0])
print("first target:", round(y_train[0], 2))

train: 240 rows | test: 60 rows
first record: {'sqft': 912.0, 'bedrooms': 4, 'age': 51.1}
first target: 187.89


---
## Step 2 — From dicts to a matrix (and the key-order trap)

Linear algebra needs a **matrix**, not dicts. So we need a rule that maps
`{"sqft": 1200, "bedrooms": 3, "age": 10}` to the row `[10.0, 3.0, 1200.0]`.

The rule must be **frozen at `fit` time**, because dictionaries do not guarantee a shared key
order across rows:

```python
a = {"sqft": 1200, "bedrooms": 3}
b = {"bedrooms": 3, "sqft": 1200}   # same house, different key order
list(a.values())  ->  [1200, 3]
list(b.values())  ->  [3, 1200]     # <-- silently wrong
```

Two decisions we commit to:

1. **Feature order = `sorted(union of all keys seen at fit time)`.** Deterministic, and
   independent of how any individual dict was built.
2. **A missing key is a `ValueError`, not a zero.** Scikit-learn's `DictVectorizer` silently
   fills missing features with `0`, which is indistinguishable from a genuine measurement of
   zero. We refuse to guess.

In [3]:
def extract_feature_names(X):
    # Union of every key seen, sorted -> a deterministic, reproducible feature order.
    names = set()
    for row in X:
        names.update(row.keys())
    return sorted(names)


def records_to_matrix(X, feature_names):
    # Convert list[dict] -> (n_samples, n_features) float array using a FIXED feature order.
    known = set(feature_names)
    out = np.empty((len(X), len(feature_names)), dtype=float)

    for i, row in enumerate(X):
        missing = [f for f in feature_names if f not in row]
        if missing:
            raise ValueError(
                f"record {i} is missing feature(s) {missing}. "
                f"Expected exactly {feature_names}. "
                f"A missing feature is not the same as a zero -- fill it explicitly if you mean 0."
            )
        unknown = [k for k in row if k not in known]
        if unknown:
            raise ValueError(
                f"record {i} has unknown feature(s) {unknown}. "
                f"The model was fitted on {feature_names}."
            )
        out[i] = [row[f] for f in feature_names]

    return out


feature_names = extract_feature_names(X_train)
print("feature order:", feature_names)

M = records_to_matrix(X_train[:3], feature_names)
print(M)

feature order: ['age', 'bedrooms', 'sqft']
[[  51.1    4.   912. ]
 [  45.1    1.   848. ]
 [  51.8    3.  3297. ]]


In [4]:
# The two guards, demonstrated.
for bad, label in [
    ([{"sqft": 1200.0, "bedrooms": 3}],                        "missing 'age'"),
    ([{"sqft": 1200.0, "bedrooms": 3, "age": 10.0, "pool": 1}], "unknown 'pool'"),
]:
    try:
        records_to_matrix(bad, feature_names)
    except ValueError as e:
        print(f"[{label}] ValueError: {e}\n")

[missing 'age'] ValueError: record 0 is missing feature(s) ['age']. Expected exactly ['age', 'bedrooms', 'sqft']. A missing feature is not the same as a zero -- fill it explicitly if you mean 0.

[unknown 'pool'] ValueError: record 0 has unknown feature(s) ['pool']. The model was fitted on ['age', 'bedrooms', 'sqft'].



In [5]:
# And the trap it protects us from: key order does NOT matter, by construction.
same_house_a = {"sqft": 1200.0, "bedrooms": 3, "age": 10.0}
same_house_b = {"age": 10.0, "sqft": 1200.0, "bedrooms": 3}

print("naive .values():", list(same_house_a.values()), "vs", list(same_house_b.values()))
print("our converter  :", records_to_matrix([same_house_a], feature_names)[0],
      "vs", records_to_matrix([same_house_b], feature_names)[0])

naive .values(): [1200.0, 3, 10.0] vs [10.0, 1200.0, 3]
our converter  : [  10.    3. 1200.] vs [  10.    3. 1200.]


---
## Step 3 — The model

A linear model is just a dot product plus a bias:

$$\hat{y} = \mathbf{w} \cdot \mathbf{x} + b$$

`w` is a vector with one weight per feature (`coef_` in scikit-learn), `b` is a single number
(`intercept_`). That is the *entire* model. Everything after this step is about **finding good
values for `w` and `b`**.

In [6]:
def predict_matrix(X_mat, coef, intercept):
    return X_mat @ coef + intercept


# With the TRUE coefficients, the prediction should be very close to the true price.
true_coef_vec = np.array([TRUE_COEF[f] for f in feature_names])   # in sorted feature order!
print("feature order:", feature_names)
print("coef vector  :", true_coef_vec)

X_mat_train = records_to_matrix(X_train, feature_names)
y_arr_train = np.asarray(y_train)

oracle = predict_matrix(X_mat_train, true_coef_vec, TRUE_INTERCEPT)
print("\npredicted:", oracle[:4].round(1))
print("actual   :", y_arr_train[:4].round(1))
print("residual sd:", (oracle - y_arr_train).std().round(2), "(we injected noise sd=15)")

feature order: ['age', 'bedrooms', 'sqft']
coef vector  : [-0.8  10.    0.15]

predicted: [185.9 151.1 533.1 527.3]
actual   : [187.9 151.1 525.6 511.6]
residual sd: 15.25 (we injected noise sd=15)


---
## Step 4 — Loss and gradient for **one** sample

Scikit-learn's default loss is `squared_error`, defined per sample as

$$L(\mathbf{w}, b) = \tfrac{1}{2}\,(\hat{y} - y)^2$$

The $\tfrac{1}{2}$ is a convenience: it cancels when we differentiate.

$$\frac{\partial L}{\partial \mathbf{w}} = (\hat{y} - y)\,\mathbf{x}
\qquad
\frac{\partial L}{\partial b} = (\hat{y} - y)$$

This is the whole of "SGD" — **one sample, one gradient, one update**. Read the weight
gradient out loud: *error times the input*. If the error is positive (we over-predicted) and a
feature is positive, we push that weight down.

With L2 regularisation (`penalty="l2"`, strength `alpha`) we add $\alpha\mathbf{w}$ to the
weight gradient. The intercept is **not** regularised — penalising it would bias the model
toward predicting near zero for no good reason.

In [7]:
def sample_gradient(x, y, coef, intercept, alpha=0.0):
    # Returns (loss, grad_w, grad_b) for a SINGLE sample.
    pred = float(x @ coef + intercept)
    err = pred - y
    loss = 0.5 * err ** 2
    grad_w = err * x + alpha * coef     # alpha*coef = d/dw of (alpha/2)*||w||^2
    grad_b = err                        # intercept is NOT regularised
    return loss, grad_w, grad_b


coef = np.zeros(len(feature_names))
intercept = 0.0

loss, gw, gb = sample_gradient(X_mat_train[0], y_arr_train[0], coef, intercept)
print("sample x   :", X_mat_train[0])
print("sample y   :", round(y_arr_train[0], 2))
print("loss       :", round(loss, 2))
print("grad_w     :", gw)
print("grad_b     :", round(gb, 2))

sample x   : [ 51.1   4.  912. ]
sample y   : 187.89
loss       : 17650.55
grad_w     : [  -9600.9678    -751.5435 -171351.9111]
grad_b     : -187.89


In [8]:
# Gradient check: compare the analytic gradient against a numerical one.
# If these disagree, the maths above is wrong -- always worth 5 lines.
def numeric_grad(x, y, coef, intercept, alpha=0.0, h=1e-6):
    g = np.zeros_like(coef)
    for j in range(len(coef)):
        up, dn = coef.copy(), coef.copy()
        up[j] += h
        dn[j] -= h
        f_up = 0.5 * (float(x @ up + intercept) - y) ** 2 + 0.5 * alpha * up @ up
        f_dn = 0.5 * (float(x @ dn + intercept) - y) ** 2 + 0.5 * alpha * dn @ dn
        g[j] = (f_up - f_dn) / (2 * h)
    return g


w0 = np.array([0.3, -0.2, 0.05])
_, analytic, _ = sample_gradient(X_mat_train[0], y_arr_train[0], w0, 1.0, alpha=0.1)
numerical = numeric_grad(X_mat_train[0], y_arr_train[0], w0, 1.0, alpha=0.1)

print("analytic :", analytic)
print("numerical:", numerical)
print("max abs diff:", np.abs(analytic - numerical).max())

analytic : [  -6477.1948    -507.0435 -115601.3461]
numerical: [  -6477.1948    -507.0435 -115601.3461]
max abs diff: 9.751150855663582e-07


---
## Step 5 — One epoch

An **epoch** is one pass over the whole training set, sample by sample. After each sample we
take a step:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \nabla_{\mathbf{w}} L
\qquad
b \leftarrow b - \eta \cdot \nabla_b L$$

$\eta$ (`eta`) is the **learning rate** — how far we move. `shuffle=True` (scikit-learn's
default) reshuffles the order each epoch, so the model does not learn an artefact of the row
ordering.

Notice that we are **not** standardising the features yet. Watch what happens.

In [9]:
def run_one_epoch(X_mat, y, coef, intercept, eta=0.01, alpha=0.0, rng=None):
    order = np.arange(len(y)) if rng is None else rng.permutation(len(y))
    for i in order:
        _, gw, gb = sample_gradient(X_mat[i], y[i], coef, intercept, alpha)
        coef = coef - eta * gw
        intercept = intercept - eta * gb
    return coef, intercept


coef = np.zeros(len(feature_names))
intercept = 0.0

# Watch the first five individual updates before running a whole epoch.
with np.errstate(over="ignore", invalid="ignore"):
    for step in range(5):
        _, gw, gb = sample_gradient(X_mat_train[step], y_arr_train[step], coef, intercept)
        coef = coef - 0.01 * gw
        intercept = intercept - 0.01 * gb
        print(f"update {step}: largest |weight| = {np.abs(coef).max():.3e}")

update 0: largest |weight| = 1.714e+03
update 1: largest |weight| = 1.236e+07
update 2: largest |weight| = 1.344e+12
update 3: largest |weight| = 1.274e+17
update 4: largest |weight| = 4.667e+21


In [10]:
coef = np.zeros(len(feature_names))
intercept = 0.0
r = np.random.default_rng(0)

with np.errstate(over="ignore", invalid="ignore"):   # overflow is the point of this demo
    for epoch in range(3):
        coef, intercept = run_one_epoch(X_mat_train, y_arr_train, coef, intercept, eta=0.01, rng=r)
        print(f"epoch {epoch}: coef={coef}, intercept={intercept}")

epoch 0: coef=[nan nan nan], intercept=nan
epoch 1: coef=[nan nan nan], intercept=nan
epoch 2: coef=[nan nan nan], intercept=nan


It exploded. This is **not** a bug in the update rule — it is a scale problem, and it is the
single most common reason a hand-written SGD "does not work".

`sqft` is in the thousands while `bedrooms` is around 3. Starting from `w = 0`, the first
prediction is 0, so the error is the full price. For the very first training row
(`sqft = 912`, `y = 187.9`) the `sqft` gradient component is
$-187.9 \times 912 \approx -171{,}000$, so with $\eta = 0.01$ that single weight jumps by
about 1700 — which is exactly the `1.714e+03` printed above.

That overshoots wildly, which makes the *next* error larger, which makes the next step larger
still. The trace shows the weights growing by roughly four orders of magnitude per update
until they overflow to `inf`, and `inf - inf` gives `nan`.

Two fixes, and we will use both:

1. **Standardise the features** (Step 8) — the real fix.
2. **Decay the learning rate over time** (Step 6) — what scikit-learn does by default.

---
## Step 6 — Learning-rate schedules

`SGDRegressor` defaults to `learning_rate="invscaling"` with `eta0=0.01`, `power_t=0.25`:

$$\eta_t = \frac{\eta_0}{t^{\,\text{power\_t}}}$$

where $t$ counts **individual sample updates**, not epochs. Big steps early, careful steps
later.

> Worth knowing: `SGDClassifier` defaults to `learning_rate="optimal"` with `eta0=0.0`, while
> `SGDRegressor` defaults to `"invscaling"` with `eta0=0.01`. Same family, different defaults.

In [11]:
def learning_rate_at(t, eta0=0.01, schedule="invscaling", power_t=0.25):
    if schedule == "constant":
        return eta0
    if schedule == "invscaling":
        return eta0 / (t ** power_t)
    raise ValueError(f"unknown schedule {schedule!r}; use 'constant' or 'invscaling'")


for t in [1, 10, 100, 1_000, 10_000, 100_000]:
    print(f"t={t:>7,}  invscaling eta={learning_rate_at(t):.6f}   constant eta={learning_rate_at(t, schedule='constant'):.6f}")

t=      1  invscaling eta=0.010000   constant eta=0.010000
t=     10  invscaling eta=0.005623   constant eta=0.010000
t=    100  invscaling eta=0.003162   constant eta=0.010000
t=  1,000  invscaling eta=0.001778   constant eta=0.010000
t= 10,000  invscaling eta=0.001000   constant eta=0.010000
t=100,000  invscaling eta=0.000562   constant eta=0.010000


---
## Step 7 — The full class

Now we assemble everything behind the scikit-learn API: `__init__` stores hyper-parameters
only, `fit` learns and returns `self`, `predict` uses what was learned. Attributes learned from
data end with an underscore (`coef_`, `intercept_`, `n_iter_`).

**Stopping rule**, mirroring scikit-learn: after each epoch we compute the training loss. If it
fails to improve by at least `tol` for `n_iter_no_change` consecutive epochs, we stop.

**Output format**: `predict` returns a `list[dict]` — one dict per input record — so the output
shape mirrors the input shape.

In [12]:
class SGDRegressorDict:
    # Linear regression trained by stochastic gradient descent.
    # Input:  X as list[dict], y as list[float]
    # Output: predict -> list[dict], one {"prediction": value} per input record.

    def __init__(self, eta0=0.01, alpha=1e-4, max_iter=1000, tol=1e-3,
                 n_iter_no_change=5, learning_rate="invscaling", power_t=0.25,
                 fit_intercept=True, shuffle=True, random_state=None,
                 output_key="prediction"):
        self.eta0 = eta0
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol
        self.n_iter_no_change = n_iter_no_change
        self.learning_rate = learning_rate
        self.power_t = power_t
        self.fit_intercept = fit_intercept
        self.shuffle = shuffle
        self.random_state = random_state
        self.output_key = output_key

    # ---------- dict <-> matrix adapter ----------
    def _to_matrix(self, X):
        return records_to_matrix(X, self.feature_names_)

    # ---------- learning-rate schedule ----------
    def _eta(self, t):
        return learning_rate_at(t, self.eta0, self.learning_rate, self.power_t)

    # ---------- API ----------
    def fit(self, X, y):
        if len(X) != len(y):
            raise ValueError(f"X has {len(X)} records but y has {len(y)} targets")

        self.feature_names_ = extract_feature_names(X)
        X_mat = self._to_matrix(X)
        y_arr = np.asarray(y, dtype=float)

        if not np.isfinite(X_mat).all():
            raise ValueError("X contains NaN or inf; clean or impute before fitting")

        n_samples, n_features = X_mat.shape
        rng = np.random.default_rng(self.random_state)

        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0.0
        self.loss_curve_ = []

        t = 1
        best_loss = np.inf
        n_bad_epochs = 0

        for epoch in range(self.max_iter):
            order = rng.permutation(n_samples) if self.shuffle else np.arange(n_samples)

            for i in order:
                xi, yi = X_mat[i], y_arr[i]
                err = float(xi @ self.coef_ + self.intercept_) - yi
                eta = self._eta(t)
                self.coef_ -= eta * (err * xi + self.alpha * self.coef_)
                if self.fit_intercept:
                    self.intercept_ -= eta * err
                t += 1

            resid = (X_mat @ self.coef_ + self.intercept_) - y_arr
            epoch_loss = 0.5 * np.mean(resid ** 2)
            self.loss_curve_.append(epoch_loss)

            if not np.isfinite(epoch_loss):
                raise FloatingPointError(
                    f"loss diverged to {epoch_loss} at epoch {epoch}. "
                    f"Standardise your features or lower eta0."
                )

            if epoch_loss > best_loss - self.tol:
                n_bad_epochs += 1
            else:
                n_bad_epochs = 0
            best_loss = min(best_loss, epoch_loss)

            if n_bad_epochs >= self.n_iter_no_change:
                break

        self.n_iter_ = epoch + 1
        self.t_ = t
        return self

    def predict(self, X):
        if not hasattr(self, "coef_"):
            raise RuntimeError("call fit() before predict()")
        scores = self._to_matrix(X) @ self.coef_ + self.intercept_
        return [{self.output_key: float(s)} for s in scores]

    def score(self, X, y):
        # R^2, same definition scikit-learn uses.
        y_arr = np.asarray(y, dtype=float)
        pred = np.array([d[self.output_key] for d in self.predict(X)])
        ss_res = np.sum((y_arr - pred) ** 2)
        ss_tot = np.sum((y_arr - y_arr.mean()) ** 2)
        return 1 - ss_res / ss_tot

    @property
    def coef_dict_(self):
        # Coefficients labelled by feature name -- much easier to read than a bare array.
        return dict(zip(self.feature_names_, self.coef_))


print("class defined")

class defined


---
## Step 8 — Scaling: the fix

Raw features diverge, exactly as in Step 5. The fix is to standardise, and standardisation must
be **fitted on the training set only** — computing the mean and standard deviation over
train + test leaks test information into training.

Since our data format is `list[dict]`, the scaler works on `list[dict]` too.

> Recall that a z-score is a *linear* transformation: it changes location and scale only. The
> shape of the distribution is untouched. What it changes is the **geometry the optimiser
> sees** — after scaling, every feature contributes gradients of a comparable magnitude.

In [13]:
class DictStandardScaler:
    # z-score standardisation for list[dict], fitted on training data only.

    def fit(self, X):
        self.feature_names_ = extract_feature_names(X)
        M = records_to_matrix(X, self.feature_names_)
        self.mean_ = M.mean(axis=0)
        scale = M.std(axis=0)
        scale[scale == 0.0] = 1.0        # a constant column would divide by zero
        self.scale_ = scale
        return self

    def transform(self, X):
        M = (records_to_matrix(X, self.feature_names_) - self.mean_) / self.scale_
        return [dict(zip(self.feature_names_, row)) for row in M]

    def fit_transform(self, X):
        return self.fit(X).transform(X)


scaler = DictStandardScaler().fit(X_train)          # fit on TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)                # test uses the TRAIN statistics

print("mean_ :", {k: round(float(v), 2) for k, v in zip(scaler.feature_names_, scaler.mean_)})
print("scale_:", {k: round(float(v), 2) for k, v in zip(scaler.feature_names_, scaler.scale_)})
print("\nbefore:", X_train[0])
print("after :", {k: round(float(v), 3) for k, v in X_train_s[0].items()})

mean_ : {'age': 29.97, 'bedrooms': 3.11, 'sqft': 2129.83}
scale_: {'age': 17.18, 'bedrooms': 1.43, 'sqft': 784.08}

before: {'sqft': 912.0, 'bedrooms': 4, 'age': 51.1}
after : {'age': 1.23, 'bedrooms': 0.619, 'sqft': -1.553}


In [14]:
model = SGDRegressorDict(eta0=0.01, alpha=1e-4, random_state=0).fit(X_train_s, y_train)

print(f"stopped after {model.n_iter_} epochs ({model.t_ - 1:,} weight updates)")
print("loss curve (first 5):", [round(float(v), 1) for v in model.loss_curve_[:5]])
print("loss curve (last 3) :", [round(float(v), 3) for v in model.loss_curve_[-3:]])
print("\ncoef_     :", {k: round(float(v), 3) for k, v in model.coef_dict_.items()})
print("intercept_:", round(model.intercept_, 3))
print(f"\nR^2 train: {model.score(X_train_s, y_train):.4f}")
print(f"R^2 test : {model.score(X_test_s,  y_test):.4f}")

stopped after 35 epochs (8,400 weight updates)
loss curve (first 5): [15174.0, 5052.3, 1977.1, 877.5, 443.8]
loss curve (last 3) : [115.702, 115.701, 115.703]

coef_     : {'age': -14.248, 'bedrooms': 14.282, 'sqft': 116.71}
intercept_: 376.391

R^2 train: 0.9837
R^2 test : 0.9869


Those coefficients are in **standardised units** — "price change per one standard deviation of
the feature". To compare against the true rule we un-scale them:

$$\beta_j^{\text{raw}} = \frac{w_j}{\text{scale}_j}
\qquad
b^{\text{raw}} = b - \sum_j \frac{w_j \cdot \text{mean}_j}{\text{scale}_j}$$

In [15]:
raw_coef = model.coef_ / scaler.scale_
raw_intercept = model.intercept_ - np.sum(model.coef_ * scaler.mean_ / scaler.scale_)

comparison = pd.DataFrame({
    "recovered": dict(zip(model.feature_names_, raw_coef)),
    "true":      TRUE_COEF,
})
print(comparison)
print(f"\nintercept  recovered={raw_intercept:.2f}   true={TRUE_INTERCEPT:.2f}")

          recovered   true
age          -0.829  -0.80
bedrooms      9.955  10.00
sqft          0.149   0.15

intercept  recovered=53.23   true=50.00


---
## Step 9 — Sanity check against the real `SGDRegressor`

Our implementation is a faithful *teaching* version, not a bit-for-bit clone: scikit-learn's
Cython inner loop uses a scaled weight-vector representation for efficiency and applies a few
extra tricks. What should match is the **destination**, not the exact path — similar
coefficients and similar $R^2$.

`predict` returns a `list[dict]`, so we pull the values out before scoring.

In [16]:
from sklearn.linear_model import SGDRegressor

X_train_arr = records_to_matrix(X_train_s, feature_names)
X_test_arr  = records_to_matrix(X_test_s,  feature_names)

sk = SGDRegressor(loss="squared_error", penalty="l2", alpha=1e-4, eta0=0.01,
                  learning_rate="invscaling", power_t=0.25,
                  max_iter=1000, tol=1e-3, n_iter_no_change=5,
                  shuffle=True, random_state=0)
sk.fit(X_train_arr, y_train)

side_by_side = pd.DataFrame({
    "ours":    dict(zip(model.feature_names_, model.coef_)),
    "sklearn": dict(zip(feature_names, sk.coef_)),
})
print(side_by_side)
print(f"\nintercept   ours={model.intercept_:.3f}   sklearn={sk.intercept_[0]:.3f}")
print(f"epochs      ours={model.n_iter_}          sklearn={sk.n_iter_}")
print(f"\nR^2 test    ours={model.score(X_test_s, y_test):.4f}   sklearn={sk.score(X_test_arr, y_test):.4f}")

             ours  sklearn
age       -14.248  -14.230
bedrooms   14.282   14.355
sqft      116.710  116.687

intercept   ours=376.391   sklearn=376.266
epochs      ours=35          sklearn=30

R^2 test    ours=0.9869   sklearn=0.9869


In [17]:
# The declared output contract: list[dict], one dict per input record.
preds = model.predict(X_test_s)

print(type(preds), "of", type(preds[0]))
print("length:", len(preds), "== number of input records:", len(X_test_s))
for p, actual in zip(preds[:5], y_test[:5]):
    print(f"  {{'prediction': {p['prediction']:.2f}}}   actual={actual:.2f}")

<class 'list'> of <class 'dict'>
length: 60 == number of input records: 60
  {'prediction': 287.20}   actual=284.97
  {'prediction': 234.09}   actual=231.18
  {'prediction': 268.70}   actual=270.87
  {'prediction': 191.78}   actual=200.08
  {'prediction': 408.94}   actual=410.79


---
# Part 2 — The pandas version

Everything above works on `list[dict]`. Now we switch both ends of the pipe to pandas.

## Step 10 — The bridge

The two formats convert into each other in one call, which is why `list[dict]` is such a
common API payload format:

| direction | call |
|---|---|
| `list[dict]` → `DataFrame` | `pd.DataFrame(records)` |
| `DataFrame` → `list[dict]` | `df.to_dict(orient="records")` |

In [18]:
df_train = pd.DataFrame(X_train)
df_test  = pd.DataFrame(X_test)
s_train  = pd.Series(y_train, name="price_k")
s_test   = pd.Series(y_test,  name="price_k")

print(df_train.head(3))
print("\ncolumns:", list(df_train.columns), " <- INSERTION order, not sorted")
print("dtypes:\n", df_train.dtypes.to_string())

# round trip
back = df_train.to_dict(orient="records")
print("\nround trip identical:", back[:1] == X_train[:1])

     sqft  bedrooms   age
0   912.0         4  51.1
1   848.0         1  45.1
2  3297.0         3  51.8

columns: ['sqft', 'bedrooms', 'age']  <- INSERTION order, not sorted
dtypes:
 sqft        float64
bedrooms      int64
age         float64

round trip identical: True


Note the column order: `pd.DataFrame(records)` uses the **insertion order of the keys**
(`sqft, bedrooms, age`), whereas our dict class sorted them (`age, bedrooms, sqft`). Neither is
wrong — but each class must be internally consistent, which is why the order is captured at
`fit` time in both designs.

---
## Step 11 — `DataFrame` input, and the column-order trap

With a DataFrame, the column names give us the feature order for free. The temptation is to
write `df.values` and be done. That is exactly the dict key-order trap wearing a different hat:

```python
model.fit(df[["sqft", "bedrooms", "age"]])
model.predict(df[["age", "sqft", "bedrooms"]])   # .values -> silently wrong
```

Nothing raises. The shapes match. The predictions are simply wrong. Let's watch it happen.

In [19]:
naive_coef = model.coef_                # trained in order ['age', 'bedrooms', 'sqft']
naive_intercept = model.intercept_

# An upstream service hands you the same rows, with the columns in ITS preferred order.
df_upstream = pd.DataFrame(X_test_s)[["sqft", "bedrooms", "age"]]

correct = df_upstream[model.feature_names_].to_numpy()   # reindexed to the FITTED order
naive   = df_upstream.to_numpy()                         # raw .values, WRONG order


def r2(pred, y):
    y = np.asarray(y, dtype=float)
    return 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)


print("fitted feature order  :", model.feature_names_)
print("incoming column order :", list(df_upstream.columns))
print("\nshapes match, so nothing complains:", correct.shape, "==", naive.shape)
print(f"\nR^2 with reindexing : {r2(correct @ naive_coef + naive_intercept, y_test):+.4f}")
print(f"R^2 with .to_numpy(): {r2(naive   @ naive_coef + naive_intercept, y_test):+.4f}   <- no error raised")

fitted feature order  : ['age', 'bedrooms', 'sqft']
incoming column order : ['sqft', 'bedrooms', 'age']

shapes match, so nothing complains: (60, 3) == (60, 3)

R^2 with reindexing : +0.9869
R^2 with .to_numpy(): -2.2627   <- no error raised


A negative $R^2$ means the model is worse than predicting the mean. And nothing warned us.

The fix is one line — **reindex the DataFrame to the fitted column order before converting to
an array** — plus explicit errors for missing and unexpected columns, mirroring the dict version.

In [20]:
def frame_to_matrix(df, feature_names):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"expected a DataFrame, got {type(df).__name__}")

    missing = [c for c in feature_names if c not in df.columns]
    if missing:
        raise ValueError(f"DataFrame is missing column(s) {missing}; expected {feature_names}")

    unknown = [c for c in df.columns if c not in feature_names]
    if unknown:
        raise ValueError(f"DataFrame has unexpected column(s) {unknown}; model was fitted on {feature_names}")

    M = df[feature_names].to_numpy(dtype=float)        # reindex, THEN convert
    if not np.isfinite(M).all():
        bad = df.columns[df.isna().any()].tolist()
        raise ValueError(f"non-finite values in column(s) {bad}")
    return M


for order in (["sqft", "bedrooms", "age"], ["age", "sqft", "bedrooms"], ["bedrooms", "age", "sqft"]):
    M = frame_to_matrix(df_upstream[order], model.feature_names_)
    print(f"columns {str(order):<36} R^2 = {r2(M @ naive_coef + naive_intercept, y_test):.4f}")

for bad, label in [
    (df_upstream.drop(columns=["age"]),  "missing column"),
    (df_upstream.assign(pool=1),         "unexpected column"),
]:
    try:
        frame_to_matrix(bad, model.feature_names_)
    except ValueError as e:
        print(f"\n[{label}] ValueError: {e}")

columns ['sqft', 'bedrooms', 'age']          R^2 = 0.9869
columns ['age', 'sqft', 'bedrooms']          R^2 = 0.9869
columns ['bedrooms', 'age', 'sqft']          R^2 = 0.9869

[missing column] ValueError: DataFrame is missing column(s) ['age']; expected ['age', 'bedrooms', 'sqft']

[unexpected column] ValueError: DataFrame has unexpected column(s) ['pool']; model was fitted on ['age', 'bedrooms', 'sqft']


---
## Step 12 — `DataFrame` output

For the pandas version, `predict` should return a `DataFrame` rather than a `list[dict]`. Two
things matter:

1. **Preserve the index.** If the caller passes rows with index `[7, 42, 103]`, the predictions
   must come back with the same labels — otherwise joining results back to the source data
   silently misaligns them.
2. **Return a `DataFrame`, not a `Series`**, so extra columns (prediction intervals, model
   version, a residual) can be added later without changing the return type.

---
## Step 13 — One core, two adapters

The learning algorithm has nothing to do with dicts or DataFrames. So we separate them:

```
        _SGDCore          <- the algorithm: matrices in, matrices out
       /        \
SGDRegressorDict   SGDRegressorFrame     <- adapters: I/O format only
```

Each adapter supplies three things: how to read feature names, how to build a matrix, and how
to shape the output. Everything else is inherited.

In [21]:
class _SGDCore:
    # The algorithm. Knows about matrices only -- never about dicts or DataFrames.

    def __init__(self, eta0=0.01, alpha=1e-4, max_iter=1000, tol=1e-3,
                 n_iter_no_change=5, learning_rate="invscaling", power_t=0.25,
                 fit_intercept=True, shuffle=True, random_state=None):
        self.eta0 = eta0
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol
        self.n_iter_no_change = n_iter_no_change
        self.learning_rate = learning_rate
        self.power_t = power_t
        self.fit_intercept = fit_intercept
        self.shuffle = shuffle
        self.random_state = random_state

    # --- subclasses (adapters) must provide these three ---
    def _feature_names(self, X):      raise NotImplementedError
    def _matrix(self, X):             raise NotImplementedError
    def _wrap_output(self, X, preds): raise NotImplementedError

    def _fit_matrix(self, X_mat, y_arr):
        n_samples, n_features = X_mat.shape
        rng = np.random.default_rng(self.random_state)
        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0.0
        self.loss_curve_ = []

        t, best_loss, n_bad = 1, np.inf, 0
        for epoch in range(self.max_iter):
            order = rng.permutation(n_samples) if self.shuffle else np.arange(n_samples)
            for i in order:
                xi, yi = X_mat[i], y_arr[i]
                err = float(xi @ self.coef_ + self.intercept_) - yi
                eta = learning_rate_at(t, self.eta0, self.learning_rate, self.power_t)
                self.coef_ -= eta * (err * xi + self.alpha * self.coef_)
                if self.fit_intercept:
                    self.intercept_ -= eta * err
                t += 1

            resid = (X_mat @ self.coef_ + self.intercept_) - y_arr
            epoch_loss = 0.5 * np.mean(resid ** 2)
            self.loss_curve_.append(epoch_loss)
            if not np.isfinite(epoch_loss):
                raise FloatingPointError(
                    f"loss diverged at epoch {epoch}; standardise features or lower eta0")

            n_bad = n_bad + 1 if epoch_loss > best_loss - self.tol else 0
            best_loss = min(best_loss, epoch_loss)
            if n_bad >= self.n_iter_no_change:
                break

        self.n_iter_ = epoch + 1
        return self

    def fit(self, X, y):
        self.feature_names_in_ = self._feature_names(X)
        X_mat = self._matrix(X)
        y_arr = np.asarray(y, dtype=float).ravel()
        if len(X_mat) != len(y_arr):
            raise ValueError(f"X has {len(X_mat)} rows but y has {len(y_arr)} targets")
        self.n_features_in_ = X_mat.shape[1]
        return self._fit_matrix(X_mat, y_arr)

    def predict(self, X):
        if not hasattr(self, "coef_"):
            raise RuntimeError("call fit() before predict()")
        preds = self._matrix(X) @ self.coef_ + self.intercept_
        return self._wrap_output(X, preds)

    @property
    def coef_dict_(self):
        return dict(zip(self.feature_names_in_, self.coef_))


print("core defined")

core defined


In [22]:
class SGDRegressorDictIO(_SGDCore):
    # list[dict] in, list[dict] out.

    def _feature_names(self, X):
        return extract_feature_names(X)

    def _matrix(self, X):
        return records_to_matrix(X, self.feature_names_in_)

    def _wrap_output(self, X, preds):
        return [{"prediction": float(p)} for p in preds]


class SGDRegressorFrame(_SGDCore):
    # DataFrame in, DataFrame out. Index is preserved.

    def _feature_names(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"expected a DataFrame, got {type(X).__name__}")
        return list(X.columns)

    def _matrix(self, X):
        return frame_to_matrix(X, self.feature_names_in_)

    def _wrap_output(self, X, preds):
        return pd.DataFrame({"prediction": preds}, index=X.index)

    def score(self, X, y):
        y_arr = np.asarray(y, dtype=float)
        pred = self.predict(X)["prediction"].to_numpy()
        return 1 - np.sum((y_arr - pred) ** 2) / np.sum((y_arr - y_arr.mean()) ** 2)


print("adapters defined")

adapters defined


In [23]:
# Same data, same hyper-parameters, two I/O formats.
df_train_s = pd.DataFrame(X_train_s)
df_test_s  = pd.DataFrame(X_test_s, index=[f"house_{i}" for i in test_i])   # non-default index

m_dict  = SGDRegressorDictIO(random_state=0).fit(X_train_s, y_train)
m_frame = SGDRegressorFrame(random_state=0).fit(df_train_s, s_train)

print("dict  feature order:", m_dict.feature_names_in_)
print("frame feature order:", m_frame.feature_names_in_)
print("\ncoef_dict_ (dict) :", {k: round(float(v), 4) for k, v in m_dict.coef_dict_.items()})
print("coef_dict_ (frame):", {k: round(float(v), 4) for k, v in m_frame.coef_dict_.items()})
print("\nSame model despite different internal column order:",
      np.allclose(sorted(m_dict.coef_), sorted(m_frame.coef_)))

dict  feature order: ['age', 'bedrooms', 'sqft']
frame feature order: ['age', 'bedrooms', 'sqft']

coef_dict_ (dict) : {'age': -14.248, 'bedrooms': 14.2815, 'sqft': 116.7104}
coef_dict_ (frame): {'age': -14.248, 'bedrooms': 14.2815, 'sqft': 116.7104}

Same model despite different internal column order: True


In [24]:
out_dict  = m_dict.predict(X_test_s)
out_frame = m_frame.predict(df_test_s)

print("--- list[dict] output ---")
print([{k: round(v, 2) for k, v in d.items()} for d in out_dict[:3]])

print("\n--- DataFrame output (index preserved) ---")
print(out_frame.head(3))
print(f"\nR^2 test: {m_frame.score(df_test_s, y_test):.4f}")

--- list[dict] output ---
[{'prediction': 287.2}, {'prediction': 234.09}, {'prediction': 268.7}]

--- DataFrame output (index preserved) ---
           prediction
house_107     287.202
house_90      234.091
house_15      268.698

R^2 test: 0.9869


In [25]:
# Because the index survives, results join straight back onto the source rows.
result = df_test_s.join(out_frame).assign(
    actual=y_test,
    residual=lambda d: d["actual"] - d["prediction"],
)
print(result.head())
print(f"\nresidual mean={result['residual'].mean():.3f}  sd={result['residual'].std():.3f}")

             age  bedrooms   sqft  prediction   actual  residual
house_107 -0.033    -0.078 -0.759     287.202  284.966    -2.236
house_90   1.707     1.316 -1.172     234.091  231.179    -2.912
house_15   0.700     0.619 -0.913     268.698  270.872     2.174
house_286 -1.022    -1.473 -1.526     191.781  200.081     8.300
house_153  0.770     1.316  0.212     408.943  410.785     1.842

residual mean=1.104  sd=13.904


---
## Recap

**The algorithm** — five lines that matter:

```python
err  = x @ w + b - y            # error on ONE sample
eta  = eta0 / t ** power_t      # invscaling schedule
w   -= eta * (err * x + alpha * w)
b   -= eta * err
t   += 1
```

**The traps**

| Trap | Symptom | Fix |
|---|---|---|
| Unscaled features | loss → `inf`, `nan` weights | standardise, fitted on train only |
| Dict key order | wrong values, no error | freeze `sorted(keys)` at `fit` |
| DataFrame column order | negative $R^2$, no error | reindex to `feature_names_in_` before `.to_numpy()` |
| Missing key filled with 0 | plausible but wrong predictions | raise `ValueError` |
| Scaler fitted on all data | optimistic test score | `fit` on train, `transform` test |
| Dropped index on output | misaligned joins | `pd.DataFrame(..., index=X.index)` |

**The design** — the algorithm never learns what a dict is. `_SGDCore` handles matrices;
adapters handle I/O. Adding a third format (JSON lines, Arrow, a database cursor) means writing
three small methods, not touching the optimiser.

**Exercises**

1. Add `loss="huber"` — the gradient is clipped to `epsilon` when `|err|` is large. Which of the
   two output formats has to change? (Answer: neither.)
2. Add `early_stopping=True`: hold out `validation_fraction` of the training data and use its
   loss for the stopping rule instead of the training loss.
3. Add `partial_fit(X, y)` so the model can be updated on a new batch of records without
   restarting — this is the reason SGD exists.
4. `SGDRegressorFrame` currently takes column order from the training DataFrame. Change it to
   `sorted(X.columns)` and confirm the predictions are unchanged.